In [18]:
import pandas as pd
from itertools import product
import numpy as np
import os

In [19]:
curr_dir_path = os.getcwd()
main_dir_path = os.path.abspath(os.path.join(curr_dir_path, os.pardir))
data_dir_path = os.path.join(main_dir_path, 'data')
raw_data_dir_path = os.path.join(data_dir_path, 'raw_data')
processed_data_dir_path = os.path.join(data_dir_path, 'processed_data')
cw_dir_path = os.path.join(data_dir_path, 'cw')
emissions_data_dir_path = os.path.join(raw_data_dir_path, 'emissions_data')
iea_data_dir_path = os.path.join(raw_data_dir_path, 'IEA_climate_policy_data')

In [20]:
df_policies = pd.read_csv(os.path.join(processed_data_dir_path, 'IEA_policies_clean.csv'))

# Add policy_id column
df_policies['policy_id'] = df_policies.index

df_policies

,year,jurisdiction,title,description,status,iso3,country,topic,type,category,policy_id
0,2011,National,Law 47-09 on Energy Efficiency,"Law 47-09, or the ""Law on Energy Efficiency"" a...",In force,MAR,Morocco,Unknown,Unknown,Unknown,0
1,2017,National,Swiss Energy Strategy 2050,Swiss Energy Strategy 2050The Swiss Energy Str...,In force,CHE,Switzerland,Economy-wide,"Master Energy Plan,Framework legislation",Unknown,1
2,2010,National,A Group of Energy Efficiency Measures in Agric...,This group consists of five PAMs.1. Biomass bo...,In force,FIN,Finland,Unknown,Unknown,Unknown,2
3,1999,National,New Buses,Financial assistance to regions and municipali...,In force,ITA,Italy,Unknown,Unknown,Unknown,3
4,2000,City/Municipal,New Subway System,Athens new subway system was inaugurated in Ja...,In force,GRC,Greece,Unknown,Unknown,Unknown,4
...,...,...,...,...,...,...,...,...,...,...,...
9267,2004,National,Investment Grants for SMEs and non-industrial ...,"On 30th June 2004, a law providing wide-rangi...",In force,LUX,Luxembourg,Unknown,Unknown,Unknown,9267
9268,2006,National,Energy Review,"Launched in January 2006, the Energy Review ex...",In force,GBR,United Kingdom,Unknown,Unknown,Unknown,9268
9269,1996,National,Voluntary Energy Audits,The Grand Ducal regulation of 11 August 1996 o...,In force,LUX,Luxembourg,Unknown,Unknown,Unknown,9269
9270,2006,National,Public Transit Capital Trust,Canada's Federal Budget 2006 dedicated CAD 1.3...,In force,CAN,Canada,Unknown,Unknown,Unknown,9270


In [21]:
df_policies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9272 entries, 0 to 9271
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          9272 non-null   int64 
 1   jurisdiction  9272 non-null   object
 2   title         9272 non-null   object
 3   description   9272 non-null   object
 4   status        9272 non-null   object
 5   iso3          9272 non-null   object
 6   country       9272 non-null   object
 7   topic         9272 non-null   object
 8   type          9272 non-null   object
 9   category      9272 non-null   object
 10  policy_id     9272 non-null   int64 
dtypes: int64(2), object(9)
memory usage: 796.9+ KB


In [22]:
df_emissions = pd.read_csv(os.path.join(processed_data_dir_path, 'total_emissions_db.csv'))
df_emissions

,Code,Income group,Year,Total Emissions
0,ABW,High income,2000,0.335765
1,ABW,High income,2001,0.344135
2,ABW,High income,2002,0.363222
3,ABW,High income,2003,0.412246
4,ABW,High income,2004,0.430187
...,...,...,...,...
4480,ZWE,Lower middle income,2018,47.509033
4481,ZWE,Lower middle income,2019,46.442562
4482,ZWE,Lower middle income,2020,44.576343
4483,ZWE,Lower middle income,2021,45.759664


## Helper Functions to create Features

In [23]:
def get_number_of_policies(df):
    """
    Function to get the number of policies per year, per country
    """
    df = df.groupby(['year', 'iso3', 'country']).count().reset_index()
    df = df[['year', 'iso3', 'country','policy_id']]
    df = df.rename(columns={'policy_id': 'policies_per_year'})
    
    return df
  

In [24]:
def get_number_of_policies_in_the_last_years(df, n):
    """
    Function to get the number of policies in the last "n" years for each country.
    
    This function computes a rolling sum of the 'policies_per_year' for each country (grouped by 'iso3'),
    where the window is n years (including the current year). For the earliest years, where fewer than n records
    are available, the resulting sum will be NaN.
    
    Parameters:
        df (pd.DataFrame): DataFrame with columns 'iso3', 'year', and 'policies_per_year'.
        n (int): Number of years to aggregate over.
    
    Returns:
        pd.DataFrame: DataFrame with an additional column 'policies_last_n_years' that contains the computed rolling sum.
    """
    # Ensure the DataFrame is sorted by iso3 and year
    df_sorted = df.sort_values(['iso3', 'year']).copy()

    new_col_name = f'policies_last_{n}_years'
    
    # For each country (grouped by iso3), compute the rolling sum over the last n years,
    # using min_periods=n to ensure that if there are fewer than n years, the result is NaN.
    df_sorted[new_col_name] = (
        df_sorted.groupby('iso3')['policies_per_year']
                .rolling(window=n, min_periods=n)
                .sum()
                .shift(1)  # excludes current year
                .reset_index(level=0, drop=True)
    )


    # Drop NaNs
    df_sorted = df_sorted.dropna(subset=[new_col_name])

    # Make sure the new col is int
    df_sorted[new_col_name] = df_sorted[new_col_name].astype(int)

    # Reset index
    df_sorted = df_sorted.reset_index(drop=True)
    
    return df_sorted


In [25]:
def get_previous_year_change(df):
    """
    Function to get the emissions change of the previous year for each year in each country.
    For the earliest available year for a country, the previous year change is set 
    to NaN.

    Parameters:
        df (pd.DataFrame): DataFrame with columns 'Code', 'Year', and 'emissions_change'.
    
    Returns:
        pd.DataFrame: DataFrame with a new column 'prev_year_change' containing the emissions 
                      change of the previous year for each country-year.
    """
    # Sort by country code and year to ensure correct chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()
    
    # Compute previous year change using shift within each country group
    df_sorted['prev_year_change'] = (
        df_sorted.groupby('Code')['emissions_change']
                 .shift(1)
    )
    
    return df_sorted


In [26]:
def get_previous_year_emissions(df):
    """
    Function to get the emissions of the previous year for each year in each country.
    For the earliest available year for a country, the previous year emission is set 
    to the current year's emission value.

    Parameters:
        df (pd.DataFrame): DataFrame with columns 'Code', 'Year', and 'Total Emissions'.
    
    Returns:
        pd.DataFrame: DataFrame with a new column 'prev_year_emission' containing the emission 
                      of the previous year for each country-year. For the earliest year, it uses 
                      the current year's emission.
    """
    # Sort by country code and year to ensure correct chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()
    
    # Compute previous year emissions using shift within each country group
    df_sorted['prev_year_emission'] = (
        df_sorted.groupby('Code')['Total Emissions']
                 .shift(1)
    )

    
    return df_sorted


In [27]:
def get_avg_emissions_in_previous_years(df, n):
    """
    Function to get the average emissions in the previous "n" years for each country in each year.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing columns 'Code', 'Year', and 'Total Emissions'.
        n (int): Number of years to average over (using the previous n years, not including the current year).
    
    Returns:
        pd.DataFrame: A DataFrame with an additional column 'avg_emissions_prev_n_years' that contains the average 
                      emissions for the previous n years for each country-year. For the earliest year(s) where previous 
                      data is not available, the value will be NaN.
    """
    # Sort the DataFrame by 'Code' and 'Year' to ensure proper chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()

    # new col name
    new_col_name = f'avg_emissions_prev_{n}_years'

    # Use transform so the computed series aligns with the original DataFrame's index.
    df_sorted[new_col_name] = df_sorted.groupby('Code')['Total Emissions'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )

    return df_sorted



In [28]:
def get_avg_change_in_previous_years(df, n):
    """
    Function to get the average emissions change in the previous "n" years for each country in each year.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing columns 'Code', 'Year', and 'emissions_change'.
        n (int): Number of years to average over (using the previous n years, not including the current year).
    
    Returns:
        pd.DataFrame: A DataFrame with an additional column 'avg_change_prev_n_years' that contains the average 
                      emissions change for the previous n years for each country-year. For the earliest year(s) where previous 
                      data is not available, the value will be NaN.
    """
    # Sort the DataFrame by 'Code' and 'Year' to ensure proper chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()

    # new col name
    new_col_name = f'avg_change_prev_{n}_years'

    # Use transform so the computed series aligns with the original DataFrame's index.
    df_sorted[new_col_name] = df_sorted.groupby('Code')['emissions_change'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )

    return df_sorted



In [29]:
def get_policy_lag(df, n):
    """
    Function to get the policy lag for each country in each year.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing columns 'iso3', 'year', and 'policies_last_n_years'.
        n (int): Number of years to consider for the policy lag.
    
    Returns:
        pd.DataFrame: DataFrame with an additional column 'policy_lag' that contains the policy lag for each country-year.
                      The policy lag is defined as the number of years since the last policy was implemented, up to a maximum
                      of n years. If a policy was implemented in the current year, the policy lag is 0.
    """
    # Sort the DataFrame by 'iso3' and 'year' to ensure proper chronological order
    df_sorted = df.sort_values(['iso3', 'year']).copy()

    # new col name
    new_col_name = f'policy_lag_{n}'
    
    # Determine the policy lag for each country-year
    df_sorted[new_col_name] = df_sorted.groupby('iso3')['policies_per_year'].shift(n)

    # Drop NaNs
    df_sorted = df_sorted.dropna(subset=[new_col_name])

    # Make sure the new col is int
    df_sorted[new_col_name] = df_sorted[new_col_name].astype(int)

    # Reset index
    df_sorted = df_sorted.reset_index(drop=True)
    
    return df_sorted

## Create Dataset for Prediction Model

In [48]:
df_policies_per_year = get_number_of_policies(df_policies)
df_policies_per_year.sort_values(['iso3', 'year'])

,year,iso3,country,policies_per_year
9,1971,AFG,Afghanistan,1
1004,2006,AFG,Afghanistan,1
2249,2019,AFG,Afghanistan,1
2464,2021,AFG,Afghanistan,1
10,1971,AGO,Angola,1
...,...,...,...,...
1576,2011,ZWE,Zimbabwe,2
1674,2012,ZWE,Zimbabwe,1
1774,2013,ZWE,Zimbabwe,1
2371,2019,ZWE,Zimbabwe,1


In [49]:
df_region_iso = df_policies_per_year[['iso3', 'country']].copy()
df_region_iso = df_region_iso.drop_duplicates()
df_region_iso.head()

,iso3,country
0,IND,India
1,NOR,Norway
2,AUT,Austria
3,NGA,Nigeria
4,ARG,Argentina


In [31]:
unique_isos = df_policies['iso3'].unique()

max_year = df_policies['year'].max()
min_year = df_policies['year'].min()
unique_years = range(min_year, max_year + 1)

# Create a df with all possible combinations of iso3 and year
combinations = list(product(unique_isos, unique_years))
df_combinations = pd.DataFrame(combinations, columns=['iso3', 'year'])

# Sort by iso3 and year
df_combinations = df_combinations.sort_values(by=['iso3', 'year']).reset_index(drop=True)
df_combinations.head()

,iso3,year
0,AFG,1948
1,AFG,1949
2,AFG,1950
3,AFG,1951
4,AFG,1952


In [52]:
df_template = pd.merge(df_combinations, df_region_iso, on=['iso3'], how='left')
df_template = pd.merge(df_template, df_policies_per_year.drop(columns='country'), on=['iso3', 'year'], how='left')

#NOTE: Assume that there were no policies in the years where the data is missing
df_template = df_template.fillna(0) 

# # Make policies per year int
df_template['policies_per_year'] = df_template['policies_per_year'].astype(int)
df_template

,iso3,year,country,policies_per_year
0,AFG,1948,Afghanistan,0
1,AFG,1949,Afghanistan,0
2,AFG,1950,Afghanistan,0
3,AFG,1951,Afghanistan,0
4,AFG,1952,Afghanistan,0
...,...,...,...,...
17491,ZWE,2024,Zimbabwe,0
17492,ZWE,2025,Zimbabwe,0
17493,ZWE,2026,Zimbabwe,0
17494,ZWE,2027,Zimbabwe,0


In [53]:
policies_in_the_last_years = get_number_of_policies_in_the_last_years(df_template, 3)
# policies_in_the_last_years = get_number_of_policies_in_the_last_years(policies_in_the_last_years, 5)

# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 1)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 2)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 3)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 4)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 5)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 6)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 7)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 8)
# policies_in_the_last_years = get_policy_lag(policies_in_the_last_years, 9)

policies_in_the_last_years

,iso3,year,country,policies_per_year,policies_last_3_years
0,AFG,1951,Afghanistan,0,0
1,AFG,1952,Afghanistan,0,0
2,AFG,1953,Afghanistan,0,0
3,AFG,1954,Afghanistan,0,0
4,AFG,1955,Afghanistan,0,0
...,...,...,...,...,...
17058,ZWE,2024,Zimbabwe,0,1
17059,ZWE,2025,Zimbabwe,0,0
17060,ZWE,2026,Zimbabwe,0,0
17061,ZWE,2027,Zimbabwe,0,0


In [54]:
policies_in_the_last_years[policies_in_the_last_years['iso3'] == 'USA'].head()

,iso3,year,country,policies_per_year,policies_last_3_years
16036,USA,1948,United States,0,0
16037,USA,1951,United States,0,0
16038,USA,1952,United States,0,0
16039,USA,1953,United States,0,0
16040,USA,1954,United States,0,0


In [55]:
policies_in_the_last_years[(policies_in_the_last_years['iso3'] == 'MEX') & (policies_in_the_last_years['year'] > 1999)]

,iso3,year,country,policies_per_year,policies_last_3_years
9924,MEX,2000,Mexico,2,2
9925,MEX,2001,Mexico,3,3
9926,MEX,2002,Mexico,1,6
9927,MEX,2003,Mexico,3,6
9928,MEX,2004,Mexico,4,7
9929,MEX,2005,Mexico,4,8
9930,MEX,2006,Mexico,1,11
9931,MEX,2007,Mexico,5,9
9932,MEX,2008,Mexico,11,10
9933,MEX,2009,Mexico,3,17


In [56]:
prev_year_emissions_df = get_previous_year_emissions(df_emissions)
prev_year_emissions_df.head()

,Code,Income group,Year,Total Emissions,prev_year_emission
0,ABW,High income,2000,0.335765,NaN
1,ABW,High income,2001,0.344135,0.335765
2,ABW,High income,2002,0.363222,0.344135
3,ABW,High income,2003,0.412246,0.363222
4,ABW,High income,2004,0.430187,0.412246


In [57]:
avg_emissions_prev_years_df = get_avg_emissions_in_previous_years(prev_year_emissions_df, 3)
avg_emissions_prev_years_df.head()

,Code,Income group,Year,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
0,ABW,High income,2000,0.335765,NaN,NaN
1,ABW,High income,2001,0.344135,0.335765,0.335765
2,ABW,High income,2002,0.363222,0.344135,0.339950
3,ABW,High income,2003,0.412246,0.363222,0.347708
4,ABW,High income,2004,0.430187,0.412246,0.373201


In [58]:
avg_emissions_prev_years_df[avg_emissions_prev_years_df['Code'] == 'MEX'].head()

,Code,Income group,Year,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
2645,MEX,Upper middle income,2000,426.618456,NaN,NaN
2646,MEX,Upper middle income,2001,431.652043,426.618456,426.618456
2647,MEX,Upper middle income,2002,453.963007,431.652043,429.135249
2648,MEX,Upper middle income,2003,479.173415,453.963007,437.411169
2649,MEX,Upper middle income,2004,487.731101,479.173415,454.929488


In [59]:
# Merge all the DataFrames
df_final_total_emissions = pd.merge(policies_in_the_last_years, avg_emissions_prev_years_df, left_on=['iso3', 'year'], right_on=['Code', 'Year'], how='inner')

# Drop Code and Year col
df_final_total_emissions = df_final_total_emissions.drop(columns=['Code', 'Year'])

df_final_total_emissions

,iso3,year,country,policies_per_year,policies_last_3_years,Income group,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
0,AFG,2000,Afghanistan,0,0,Low income,25.390391,NaN,NaN
1,AFG,2001,Afghanistan,0,0,Low income,23.723115,25.390391,25.390391
2,AFG,2002,Afghanistan,0,0,Low income,26.383509,23.723115,24.556753
3,AFG,2003,Afghanistan,0,0,Low income,27.071538,26.383509,25.165672
4,AFG,2004,Afghanistan,0,0,Low income,27.128799,27.071538,25.726054
...,...,...,...,...,...,...,...,...,...
4365,ZWE,2018,Zimbabwe,0,0,Lower middle income,47.509033,45.410983,46.390136
4366,ZWE,2019,Zimbabwe,1,0,Lower middle income,46.442562,47.509033,46.277579
4367,ZWE,2020,Zimbabwe,0,1,Lower middle income,44.576343,46.442562,46.454193
4368,ZWE,2021,Zimbabwe,1,1,Lower middle income,45.759664,44.576343,46.175979


In [60]:
# Make all cols lowercase
df_final_total_emissions.columns = df_final_total_emissions.columns.str.lower()

# Rename Total emissions to total_emissions and locate it at the end
df_final_total_emissions = df_final_total_emissions.rename(columns={'total emissions': 'total_emissions', 'income group': 'income_group'})

# # Calculate emissions change
# df_final_total_emissions['emissions_change'] = df_final_total_emissions['total_emissions'] - df_final_total_emissions['prev_year_emission']

# Reorder columns
cols = df_final_total_emissions.columns.tolist()
cols.remove('total_emissions')
cols.append('total_emissions')
df_final_total_emissions = df_final_total_emissions[cols]
df_final_total_emissions

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,prev_year_emission,avg_emissions_prev_3_years,total_emissions
0,AFG,2000,Afghanistan,0,0,Low income,NaN,NaN,25.390391
1,AFG,2001,Afghanistan,0,0,Low income,25.390391,25.390391,23.723115
2,AFG,2002,Afghanistan,0,0,Low income,23.723115,24.556753,26.383509
3,AFG,2003,Afghanistan,0,0,Low income,26.383509,25.165672,27.071538
4,AFG,2004,Afghanistan,0,0,Low income,27.071538,25.726054,27.128799
...,...,...,...,...,...,...,...,...,...
4365,ZWE,2018,Zimbabwe,0,0,Lower middle income,45.410983,46.390136,47.509033
4366,ZWE,2019,Zimbabwe,1,0,Lower middle income,47.509033,46.277579,46.442562
4367,ZWE,2020,Zimbabwe,0,1,Lower middle income,46.442562,46.454193,44.576343
4368,ZWE,2021,Zimbabwe,1,1,Lower middle income,44.576343,46.175979,45.759664


In [61]:
# Drop nans
df_final_total_emissions = df_final_total_emissions.dropna()
df_final_total_emissions

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,prev_year_emission,avg_emissions_prev_3_years,total_emissions
1,AFG,2001,Afghanistan,0,0,Low income,25.390391,25.390391,23.723115
2,AFG,2002,Afghanistan,0,0,Low income,23.723115,24.556753,26.383509
3,AFG,2003,Afghanistan,0,0,Low income,26.383509,25.165672,27.071538
4,AFG,2004,Afghanistan,0,0,Low income,27.071538,25.726054,27.128799
5,AFG,2005,Afghanistan,0,0,Low income,27.128799,26.861282,27.530896
...,...,...,...,...,...,...,...,...,...
4365,ZWE,2018,Zimbabwe,0,0,Lower middle income,45.410983,46.390136,47.509033
4366,ZWE,2019,Zimbabwe,1,0,Lower middle income,47.509033,46.277579,46.442562
4367,ZWE,2020,Zimbabwe,0,1,Lower middle income,46.442562,46.454193,44.576343
4368,ZWE,2021,Zimbabwe,1,1,Lower middle income,44.576343,46.175979,45.759664


In [62]:
# Save the final DataFrame
df_final_total_emissions.to_csv(os.path.join(processed_data_dir_path, 'total_emissions_with_policies.csv'), index=False)

## Build the emissions_change df

In [63]:
emissions_change_df = prev_year_emissions_df.copy()
emissions_change_df = emissions_change_df.dropna()
emissions_change_df = emissions_change_df[(emissions_change_df['Total Emissions'] > 0) & (emissions_change_df['prev_year_emission'] > 0)]
emissions_change_df['emissions_change'] = np.log(emissions_change_df['Total Emissions']) - np.log(emissions_change_df['prev_year_emission'])
emissions_change_df =  emissions_change_df.drop(columns=['prev_year_emission', 'Total Emissions'])
emissions_change_df

,Code,Income group,Year,emissions_change
1,ABW,High income,2001,0.024622
2,ABW,High income,2002,0.053979
3,ABW,High income,2003,0.126606
4,ABW,High income,2004,0.042601
5,ABW,High income,2005,0.072745
...,...,...,...,...
4480,ZWE,Lower middle income,2018,0.045166
4481,ZWE,Lower middle income,2019,-0.022704
4482,ZWE,Lower middle income,2020,-0.041013
4483,ZWE,Lower middle income,2021,0.026200


In [64]:
emissions_change_df = get_avg_change_in_previous_years(emissions_change_df, 3)
# emissions_change_df = get_avg_change_in_previous_years(emissions_change_df, 5)
emissions_change_df = get_previous_year_change(emissions_change_df)
emissions_change_df.head()

,Code,Income group,Year,emissions_change,avg_change_prev_3_years,prev_year_change
1,ABW,High income,2001,0.024622,NaN,NaN
2,ABW,High income,2002,0.053979,0.024622,0.024622
3,ABW,High income,2003,0.126606,0.039301,0.053979
4,ABW,High income,2004,0.042601,0.068402,0.126606
5,ABW,High income,2005,0.072745,0.074395,0.042601


In [65]:
df_final_emissions_change = pd.merge(policies_in_the_last_years, emissions_change_df, left_on=['iso3', 'year'], right_on=['Code', 'Year'], how='inner')
df_final_emissions_change = df_final_emissions_change.drop(columns=['Code', 'Year'])
df_final_emissions_change = df_final_emissions_change.dropna()

# rename Income group as income_group
df_final_emissions_change = df_final_emissions_change.rename(columns={'Income group': 'income_group'})

# Reorder columns so emissions_change is at the end
cols = df_final_emissions_change.columns.tolist()
cols.remove('emissions_change')
cols.append('emissions_change')
df_final_emissions_change = df_final_emissions_change[cols]

df_final_emissions_change

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,avg_change_prev_3_years,prev_year_change,emissions_change
1,AFG,2002,Afghanistan,0,0,Low income,-0.067921,-0.067921,0.106289
2,AFG,2003,Afghanistan,0,0,Low income,0.019184,0.106289,0.025744
3,AFG,2004,Afghanistan,0,0,Low income,0.021371,0.025744,0.002113
4,AFG,2005,Afghanistan,0,0,Low income,0.044715,0.002113,0.014713
5,AFG,2006,Afghanistan,1,0,Low income,0.014190,0.014713,0.009749
...,...,...,...,...,...,...,...,...,...
3711,ZWE,2018,Zimbabwe,0,0,Lower middle income,-0.010956,-0.010988,0.045166
3712,ZWE,2019,Zimbabwe,1,0,Lower middle income,-0.002361,0.045166,-0.022704
3713,ZWE,2020,Zimbabwe,0,1,Lower middle income,0.003825,-0.022704,-0.041013
3714,ZWE,2021,Zimbabwe,1,1,Lower middle income,-0.006184,-0.041013,0.026200


In [66]:
df_final_emissions_change.to_csv(os.path.join(processed_data_dir_path, 'emissions_change_with_policies.csv'), index=False)